In [1]:
import cv2
import numpy as np
import os

if not os.path.exists("dataset"):
  os.makedirs("dataset/yes") # if not exists, create new dataset (folder)
  os.makedirs("dataset/no") # and put yes & no (files) in it.

print("Making Data of phantom rays is Loading... 🧠")

for i in range(100):
  img = np.zeros((128, 128), dtype=np.uint8)

  center = (np.random.randint(30, 90), np.random.randint(30, 90))
  cv2.circle(img, center, 15, (255), -1)

  cv2.imwrite(f"dataset/yes/tumor_{i}.jpg", img)

for i in range(100):
  img = np.zeros((128, 128), dtype=np.uint8)

  noise = np.random.randint(0, 50, (128, 128))
  img = img + noise
  cv2.imwrite(f"dataset/no/healthy_{i}.jpg", img)

print("200 ray images is created successfully! ✅")

Making Data of phantom rays is Loading... 🧠
200 ray images is created successfully! ✅


In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import train_test_split

DATA_DIR = "dataset"
CATEGORIES = ["no", "yes"]
IMG_SIZE = 64

data = []
labels = []

print("1. Download Image and Transfering it to Digits is Loading... ⌛")

for category in CATEGORIES:
  path = os.path.join(DATA_DIR, category) 
  class_num = CATEGORIES.index(category) 

  for img_name in os.listdir(path): 
    try:
      img_path = os.path.join(path, img_name) 
      img_array = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
      new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
      data.append(new_array)
      labels.append(class_num)
    except Exception as e:
      pass

X = np.array(data).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
X = X / 255.0
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data is Loaded! Number of Train Images: {len(X_train)}, Test Images: {len(X_test)}")

1. Download Image and Transfering it to Digits is Loading... ⌛
Data is Loaded! Number of Train Images: 160, Test Images: 40


In [6]:
print("2. Artifital Brain is Loading (CNN)... 🧠")

model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=X.shape[1:])) 
model.add(MaxPooling2D(pool_size=(2, 2))) 

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

print("3. Training is Start (Model is Study Now)... 📚")
model.fit(X_train, y_train, batch_size=10, epochs=5, validation_split=0.1)

2. Artifital Brain is Loading (CNN)... 🧠
3. Training is Start (Model is Study Now)... 📚
Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 130ms/step - accuracy: 0.8750 - loss: 0.3095 - val_accuracy: 1.0000 - val_loss: 9.0186e-04
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 113ms/step - accuracy: 1.0000 - loss: 5.1203e-04 - val_accuracy: 1.0000 - val_loss: 1.5802e-07
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step - accuracy: 1.0000 - loss: 9.1971e-07 - val_accuracy: 1.0000 - val_loss: 1.0395e-08
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 1.0000 - loss: 9.6411e-08 - val_accuracy: 1.0000 - val_loss: 5.2334e-09
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 1.0000 - loss: 6.0444e-08 - val_accuracy: 1.0000 - val_loss: 4.4488e-09


In [7]:
print("4. Exam (moment of truth)... 📑")
loss, accuracy = model.evaluate(X_test, y_test)

print(f"\n====================")
print(f"Model IQ: {accuracy * 100:.2f}%")
print(f"====================\n")

model.save('brain_tumor_detector.h5')
print("Model is Saved Successfully! 📰")

4. Exam (moment of truth)... 📑
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 1.0000 - loss: 5.9354e-08 



Model IQ: 100.00%

Model is Saved Successfully! 📰


In [15]:
import random

print("Artifitial Brain is Loading... 🧠")
loaded_model = tf.keras.models.load_model('brain_tumor_detector.h5')

def prepare_image(filepath):
  IMG_SIZE = 64
  img_array = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE) 
  new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
  return new_array.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0 

random_filename = random.choice(os.listdir("dataset/yes"))
image_path = f"dataset/yes/{random_filename}"

print(f"Check Image: {random_filename} ... ⌛")
prediction = loaded_model.predict([prepare_image(image_path)]) 

print("------------------------------------------------")
print(f"Result of Checking (Confidence): {prediction[0][0]}")

if prediction[0][0] > 0.5:
  print("Tumor Detected 🚨")
else:
  print("Healthy ✅")
print("------------------------------------------------")

img = cv2.imread(image_path)
cv2.imshow("MRI Scan", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Artifitial Brain is Loading... 🧠
Check Image: tumor_84.jpg ... ⌛
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
------------------------------------------------
Result of Checking (Confidence): 1.0
Tumor Detected 🚨
------------------------------------------------
